In [4]:
import re, json, os
from collections import defaultdict, Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

os.makedirs('results', exist_ok=True)

# **2. Corpus**
The original texts

In [5]:
corpus = {
    "D1":  "Information retrieval helps users find relevant documents.",
    "D2":  "Information Retrieval systems index documents and documents.",
    "D3":  "Search engines retrieve relevant web documents.",
    "D4":  "Document indexing supports efficient information retrieval.",
    "D5":  "Machine learning improves search engine ranking.",
    "D6":  "Search systems use indexed documents to retrieve information.",
    "D7":  "The search engine retrieves documents; the documents are searchable.",
    "D8":  "Indexing documents makes information retrieval more efficient.",
    "D9":  "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections.",
}

## **Part 1 – Text preprocessing**
# A. Inspecting the corpus

In [6]:
for d, text in corpus.items():
    caps   = re.findall(r'\b[A-Z]\w*', text)
    punct  = re.findall(r'[^\w\s]', text)
    counts = Counter(re.sub(r'[^\w\s]', ' ', text.lower()).split())
    repeated = {w: c for w, c in counts.items() if c > 1}
    print(f"{d:>3} | caps: {caps} | punct: {punct} | repeated: {repeated}")

 D1 | caps: ['Information'] | punct: ['.'] | repeated: {}
 D2 | caps: ['Information', 'Retrieval'] | punct: ['.'] | repeated: {'documents': 2}
 D3 | caps: ['Search'] | punct: ['.'] | repeated: {}
 D4 | caps: ['Document'] | punct: ['.'] | repeated: {}
 D5 | caps: ['Machine'] | punct: ['.'] | repeated: {}
 D6 | caps: ['Search'] | punct: ['.'] | repeated: {}
 D7 | caps: ['The'] | punct: [';', '.'] | repeated: {'the': 2, 'documents': 2}
 D8 | caps: ['Indexing'] | punct: ['.'] | repeated: {}
 D9 | caps: ['DOCUMENTS'] | punct: ['.'] | repeated: {}
D10 | caps: ['Retrieval'] | punct: ['.'] | repeated: {}


# Observations

*   Uppercase/lowercase: every document starts with a capital, witch shows that the same word may appear in several case with different forms (documents/DOCUMENTS, Information/information).
*   Punctuation: a full stop ends each sentence; D7 also contains a semicolon.
*   Repeated terms: documents twice in D2, the and documents twice in D7.
*   Morphological variants: document / documents; index / indexed / indexing; retrieve / retrieves / retrieval; search / searchable; engine / engines; system / systems.

# B. Basic steps implemented by hand (version A)

In [7]:
def preprocess_document(text):
    """Lowercase, remove punctuation, and tokenize on whitespace."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text.split()

tokens_A = {d: preprocess_document(t) for d, t in corpus.items()}
for d, toks in tokens_A.items():
    print(f"{d:>3}: {toks}")

 D1: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
 D2: ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
 D3: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
 D4: ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
 D5: ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
 D6: ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information']
 D7: ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable']
 D8: ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient']
 D9: ['documents', 'contain', 'information', 'about', 'document', 'indexing']
D10: ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']


# C. Stop-word removal (version B)

In [8]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words]

tokens_B = {d: remove_stopwords(toks) for d, toks in tokens_A.items()}
for d, toks in tokens_B.items():
    print(f"{d:>3}: {toks}")

 D1: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
 D2: ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']
 D3: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
 D4: ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
 D5: ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
 D6: ['search', 'systems', 'use', 'indexed', 'documents', 'retrieve', 'information']
 D7: ['search', 'engine', 'retrieves', 'documents', 'documents', 'searchable']
 D8: ['indexing', 'documents', 'makes', 'information', 'retrieval', 'efficient']
 D9: ['documents', 'contain', 'information', 'document', 'indexing']
D10: ['retrieval', 'systems', 'organise', 'documents', 'document', 'collections']


# D. Porter stemming (version C)
Porter stemming is a rule-based suffix stripper, not lemmatization: its output is not always a real word.

In [9]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

tokens_C = {d: stem_tokens(toks) for d, toks in tokens_B.items()}
for d, toks in tokens_C.items():
    print(f"{d:>3}: {toks}")

groups = defaultdict(set)
for toks in tokens_B.values():
    for w in toks:
        groups[stemmer.stem(w)].add(w)

print("\nStems merging multiple word forms:")
for s, ws in sorted(groups.items()):
    if len(ws) > 1:
        print(f"  {s:12s} <- {sorted(ws)}")

versions = {'A': tokens_A, 'B': tokens_B, 'C': tokens_C}

 D1: ['inform', 'retriev', 'help', 'user', 'find', 'relev', 'document']
 D2: ['inform', 'retriev', 'system', 'index', 'document', 'document']
 D3: ['search', 'engin', 'retriev', 'relev', 'web', 'document']
 D4: ['document', 'index', 'support', 'effici', 'inform', 'retriev']
 D5: ['machin', 'learn', 'improv', 'search', 'engin', 'rank']
 D6: ['search', 'system', 'use', 'index', 'document', 'retriev', 'inform']
 D7: ['search', 'engin', 'retriev', 'document', 'document', 'searchabl']
 D8: ['index', 'document', 'make', 'inform', 'retriev', 'effici']
 D9: ['document', 'contain', 'inform', 'document', 'index']
D10: ['retriev', 'system', 'organis', 'document', 'document', 'collect']

Stems merging multiple word forms:
  document     <- ['document', 'documents']
  engin        <- ['engine', 'engines']
  index        <- ['index', 'indexed', 'indexing']
  retriev      <- ['retrieval', 'retrieve', 'retrieves']


## Part 2 – Vocabulary and document representation

In [11]:
def build_vocabulary(tokenized_documents):
    """Return sorted unique terms across the collection."""
    vocab = set()
    for toks in tokenized_documents.values():
        vocab.update(toks)
    return sorted(vocab)

def build_term_document_matrix(tokenized_documents):
    """Binary matrix: rows = sorted terms, cols = doc IDs, cell = 1 if present else 0."""
    terms = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = [
        [1 if term in tokenized_documents[d] else 0 for d in doc_ids]
        for term in terms
    ]
    return terms, doc_ids, matrix

def build_tf_matrix(tokenized_documents):
    """Term-frequency matrix: cell = number of occurrences of term in document."""
    terms = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = []
    for term in terms:
        row = []
        for d in doc_ids:
            count = 0
            for tok in tokenized_documents[d]:
                if tok == term:
                    count += 1
            row.append(count)
        matrix.append(row)
    return terms, doc_ids, matrix

def to_dataframe(terms, doc_ids, matrix):
    return pd.DataFrame(matrix, index=terms, columns=doc_ids)

vocab, bin_mat, tf_mat = {}, {}, {}
for v, toks in versions.items():
    vocab[v] = build_vocabulary(toks)
    bin_mat[v] = to_dataframe(*build_term_document_matrix(toks))
    tf_mat[v]  = to_dataframe(*build_tf_matrix(toks))
    print(f"Version {v}: vocabulary size = {len(vocab[v])}")


Version A: vocabulary size = 36
Version B: vocabulary size = 30
Version C: vocabulary size = 24


# Binary matrix and TF matrix – version A

In [12]:
print("\nBinary Matrix (Version A):")
display(bin_mat['A'])


Binary Matrix (Version A):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


In [13]:
print("\nTF Matrix (Version A):")
display(tf_mat['A'])


TF Matrix (Version A):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


# Binary matrix and TF matrix – version B

In [14]:
print("\nBinary Matrix (Version B):")
display(bin_mat['B'])


Binary Matrix (Version B):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0
find,1,0,0,0,0,0,0,0,0,0
helps,1,0,0,0,0,0,0,0,0,0
improves,0,0,0,0,1,0,0,0,0,0


In [15]:
print("\nTF Matrix (Version B):")
display(tf_mat['B'])


TF Matrix (Version B):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0
find,1,0,0,0,0,0,0,0,0,0
helps,1,0,0,0,0,0,0,0,0,0
improves,0,0,0,0,1,0,0,0,0,0


# Binary matrix and TF matrix – version C

In [16]:
print("\nBinary Matrix (Version C):")
display(bin_mat['C'])


Binary Matrix (Version C):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,1,1,1,0,1,1,1,1,1
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0


In [17]:
print("\nTF Matrix (Version C):")
display(tf_mat['C'])


TF Matrix (Version C):


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,2,1,1,0,1,2,1,2,2
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0


# Binary vs term-frequency.

*  The binary matrix only records presence/absence of a term in a document (0/1): it answers "does the document contain the term?" (Boolean retrieval).
*   The TF matrix records how many times the term occurs, so it keeps the importance of repeated words (e.g. documents = 2 in D2 and D7) – the basis of ranked retrieval (TF-IDF, BM25). The binary matrix can always be derived from the TF matrix (TF > 0), not the reverse.

## Part 3 – Inverted index with positions
# 1. Manual vs Automatic index for D1–D3 (version A)
Manual index built by hand from the token lists (zero-based positions):

* D1: information(0) retrieval(1) helps(2) users(3) find(4) relevant(5) documents(6)
* D2: information(0) retrieval(1) systems(2) index(3) documents(4) and(5) documents(6)
* D3: search(0) engines(1) retrieve(2) relevant(3) web(4) documents(5)

In [18]:
# Manual positional index for D1-D3 (Version A) on: information, retrieval, documents
manual_index_d1_d3 = {
    'information': {'D1': [0], 'D2': [0]},
    'retrieval':   {'D1': [1], 'D2': [1]},
    'documents':   {'D1': [6], 'D2': [4, 6], 'D3': [5]},
}

def build_inverted_index(tokenized_documents):
    """Build positional index: term -> {doc_id: [zero_based_positions]}."""
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, toks in tokenized_documents.items():
        for pos, term in enumerate(toks):
            index[term][doc_id].append(pos)
    return {
        term: dict(index[term])
        for term in sorted(index.keys())
    }

def search_term(inverted_index, term):
    """Return list of document IDs containing term."""
    if term not in inverted_index:
        return []
    return list(inverted_index[term].keys())

inv_index = {v: build_inverted_index(toks) for v, toks in versions.items()}

auto_subset_a = {
    t: {d: pos for d, pos in inv_index['A'].get(t, {}).items() if d in {'D1', 'D2', 'D3'}}
    for t in ['information', 'retrieval', 'documents']
}

print("Manual D1-D3 index:", manual_index_d1_d3)
print("Auto   D1-D3 index:", auto_subset_a)
print("Matches manual index?:", auto_subset_a == manual_index_d1_d3)

Manual D1-D3 index: {'information': {'D1': [0], 'D2': [0]}, 'retrieval': {'D1': [1], 'D2': [1]}, 'documents': {'D1': [6], 'D2': [4, 6], 'D3': [5]}}
Auto   D1-D3 index: {'information': {'D1': [0], 'D2': [0]}, 'retrieval': {'D1': [1], 'D2': [1]}, 'documents': {'D1': [6], 'D2': [4, 6], 'D3': [5]}}
Matches manual index?: True


## Part 4 – Effect of preprocessing

In [19]:
comparison_rows = []
num_docs = len(corpus)

for v in ['A', 'B', 'C']:
    toks_by_doc = versions[v]
    idx = inv_index[v]
    vocab_size = len(vocab[v])
    total_tokens = sum(len(toks) for toks in toks_by_doc.values())
    avg_tokens_per_doc = round(total_tokens / num_docs, 2)
    total_postings = sum(len(doc_map) for doc_map in idx.values())
    total_positions = sum(
        len(positions)
        for doc_map in idx.values()
        for positions in doc_map.values()
    )
    comparison_rows.append({
        'Version': v,
        'Vocabulary Size': vocab_size,
        'Avg Tokens / Doc': avg_tokens_per_doc,
        'Total Postings': total_postings,
        'Total Stored Positions': total_positions,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,Version,Vocabulary Size,Avg Tokens / Doc,Total Postings,Total Stored Positions
0,A,36,6.9,66,69
1,B,30,6.1,59,61
2,C,24,6.1,57,61


# Interpretation (117 words)
Preprocessing shrinks the index step by step. Lowercasing and punctuation removal (A) already merge forms such as Documents/DOCUMENTS, giving 36 terms. Stop-word removal (B) deletes frequent, low-information words (and, the, to, about): the vocabulary drops to 30, tokens per document from 6.9 to 6.1, postings from 66 to 59 and stored positions from 69 to 61. Stemming (C) does not remove tokens (still 61 positions) but merges variants (retrieval/retrieve, index/indexing, document/documents), reducing the vocabulary to 24 and postings to 57. A smaller index is cheaper and increases recall, but not every step is harmless: stop-word removal shifts positions and breaks exact phrase queries, and stemming can conflate different meanings and produce non-words (effici), so precision may fall.

## Part 5 – Verification

In [21]:
# 1 & 2. Search verification across four required cases (and B vs. C stemming effect)
search_cases = [
    ("Single-doc term (B)",        'B', 'collections'),
    ("Multi-doc term (B)",         'B', 'information'),
    ("Before stemming (B)",        'B', 'retrieval'),
    ("Unstemmed after stemming (C)", 'C', 'retrieval'),
    ("Stemmed form in C",          'C', 'retriev'),
    ("Unknown term (B)",           'B', 'unknownterm'),
]

print("=== 1-2. search_term() Verification ===")
for label, v, term in search_cases:
    res = search_term(inv_index[v], term)
    expected = [d for d, toks in versions[v].items() if term in toks]
    assert res == expected, f"Mismatch for {label} ({term}): got {res}, expected {expected}"
    print(f"[PASS] {label:30s} | query='{term}' -> {res} (matches token lists)")

# 3. Positional verification against processed token lists
print("\n=== 3. Positional Index Verification ===")
for v, toks_by_doc in versions.items():
    idx = inv_index[v]
    checked_tokens = 0
    for d, toks in toks_by_doc.items():
        for pos, tok in enumerate(toks):
            assert pos in idx[tok][d], f"Missing pos {pos} for '{tok}' in {d} (v={v})"
            checked_tokens += 1
    checked_postings = 0
    for term, doc_map in idx.items():
        for d, positions in doc_map.items():
            checked_postings += 1
            for pos in positions:
                assert toks_by_doc[d][pos] == term, f"Token mismatch at {d}[{pos}] for '{term}' (v={v})"
    print(f"[PASS] Version {v}: verified {checked_tokens} token positions across {checked_postings} postings")

# 4. Manual vs. automatic D1-D3 index comparison
print("\n=== 4. Manual vs. Automatic Index (D1-D3, Version A) ===")
discrepancies = []
for term, manual_docs in manual_index_d1_d3.items():
    auto_docs = auto_subset_a.get(term, {})
    if manual_docs != auto_docs:
        discrepancies.append((term, manual_docs, auto_docs))
assert not discrepancies, f"Discrepancies found: {discrepancies}"
print("[PASS] Zero discrepancies between manual_index_d1_d3 and auto_subset_a for 'information', 'retrieval', and 'documents'")

# 5. Binary and TF matrix verification
print("\n=== 5. Binary & TF Matrix Verification ===")
for v, toks_by_doc in versions.items():
    b_df = bin_mat[v]
    t_df = tf_mat[v]
    unique_vals = set(b_df.values.flatten())
    assert unique_vals.issubset({0, 1}), f"Invalid binary values in v={v}: {unique_vals}"
    max_tf = 0
    for term in vocab[v]:
        for d, toks in toks_by_doc.items():
            c = toks.count(term)
            max_tf = max(max_tf, c)
            assert int(t_df.loc[term, d]) == c, f"TF mismatch for '{term}' in {d} (v={v})"
            assert int(b_df.loc[term, d]) == (1 if c > 0 else 0), f"Binary mismatch for '{term}' in {d} (v={v})"
    print(f"[PASS] Version {v}: binary values in {sorted(unique_vals)}, max TF = {max_tf}, all {len(vocab[v]) * len(toks_by_doc)} cells verified")

=== 1-2. search_term() Verification ===
[PASS] Single-doc term (B)            | query='collections' -> ['D10'] (matches token lists)
[PASS] Multi-doc term (B)             | query='information' -> ['D1', 'D2', 'D4', 'D6', 'D8', 'D9'] (matches token lists)
[PASS] Before stemming (B)            | query='retrieval' -> ['D1', 'D2', 'D4', 'D8', 'D10'] (matches token lists)
[PASS] Unstemmed after stemming (C)   | query='retrieval' -> [] (matches token lists)
[PASS] Stemmed form in C              | query='retriev' -> ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D10'] (matches token lists)
[PASS] Unknown term (B)               | query='unknownterm' -> [] (matches token lists)

=== 3. Positional Index Verification ===
[PASS] Version A: verified 69 token positions across 66 postings
[PASS] Version B: verified 61 token positions across 59 postings
[PASS] Version C: verified 61 token positions across 57 postings

=== 4. Manual vs. Automatic Index (D1-D3, Version A) ===
[PASS] Zero discrepancies betw

## Part 6 – Export Files to results/

In [22]:
for v in ['A', 'B', 'C']:
    with open(f"results/tokens_{v}.json", "w", encoding="utf-8") as f:
        json.dump(versions[v], f, indent=2, ensure_ascii=False)

    with open(f"results/vocabulary_{v}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab[v]) + ("\n" if vocab[v] else ""))

    bin_mat[v].to_csv(f"results/binary_matrix_{v}.csv")
    tf_mat[v].to_csv(f"results/tf_matrix_{v}.csv")

    with open(f"results/inverted_index_{v}.json", "w", encoding="utf-8") as f:
        json.dump(inv_index[v], f, indent=2, ensure_ascii=False)

comparison_df.to_csv("results/preprocessing_comparison.csv", index=False)
print("Saved:", sorted(os.listdir('results')))

Saved: ['binary_matrix_A.csv', 'binary_matrix_B.csv', 'binary_matrix_C.csv', 'inverted_index_A.json', 'inverted_index_B.json', 'inverted_index_C.json', 'preprocessing_comparison.csv', 'tf_matrix_A.csv', 'tf_matrix_B.csv', 'tf_matrix_C.csv', 'tokens_A.json', 'tokens_B.json', 'tokens_C.json', 'vocabulary_A.txt', 'vocabulary_B.txt', 'vocabulary_C.txt']
